# 4.2 Demand Prediction

In this notebook we implement Neural Networks (NNs) to predict taxi trip demand in chicago. Additionally, we compare the performances of the NNs across different complexity levels.

For NNs there are 2 "main" complexity interpretations:
- depth: number of hidden layers
- width: number of nodes per layer
- (more complex activation function) - maybe as an extra
- (more complex optimizer) - maybe as an extra

So we decide to test 3 different NN structures:

__baseline model__:
- hidden layers: 2
- nodes per layer: 64

__wider model__:
- hidden layers: 2
- nodes per layer: 128

__deeper model__:
- hidden layers: 4
- nodes per layer: 64

For better comparison, we will test all three architectures with the same shared configurations.

Before training any NN, we establish two simple **benchmarks** to contextualise the results:

1. **Historical-mean predictor** — for each hexagon × hour-of-day pair, predict the mean `trip_count` seen in the training set. Unseen combinations fall back to the global training mean. This captures the dominant demand pattern (location + time-of-day) without any learning.
2. **Ridge regression** — a linear model fit on the same scaled feature matrix. Trained on log₁⁺ demand and back-transformed at evaluation time (same target encoding as the NNs). This shows how much a linear model can do before adding any non-linearity or depth.

In [2]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
# import matplotlib.pyplot as plt

# modeling
import copy
import random
import types
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score as _r2

# model visualization
from torchinfo import summary

# reset working dir
import os
from pathlib import Path


KeyboardInterrupt: 

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data = pd.read_parquet("data/aggregated/hexagon/demand_hex_1h_low.parquet")

In [ ]:
# How many times does each row repeat when ignoring the time bucket?
# bucket_index is also excluded as it is a numeric encoding of the time bucket.
cols_no_time = [c for c in data.columns if c not in ('time_bucket', 'bucket_index')]
counts = data[cols_no_time].value_counts()

n_total    = len(data)
n_unique   = len(counts)
n_repeated = int((counts > 1).sum())

print(f"Total rows                    : {n_total:>10,}")
print(f"Unique combinations (no time) : {n_unique:>10,}")
print(f"Combinations appearing >1x    : {n_repeated:>10,}")
print(f"Avg repetitions per combo     : {n_total / n_unique:>10.2f}x")
print(f"\nRepetition count distribution:")
dist = counts.value_counts().sort_index()
for k, v in dist.items():
    print(f"  appears {k:>3}x : {v:>8,} combinations")

# How often does each hexagon + hour-of-day combination appear?
hex_hour_counts = data.groupby(['pickup_h3_res6', 'hour_of_day']).size()

print(f"\n--- Hexagon × Hour-of-day ---")
print(f"Unique hex × hour combinations : {len(hex_hour_counts):>8,}")
print(f"Avg appearances per combo      : {hex_hour_counts.mean():>8.1f}x")
print(f"Min appearances                : {hex_hour_counts.min():>8,}")
print(f"Max appearances                : {hex_hour_counts.max():>8,}")

In [ ]:
data.head()

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Shared Configs                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# --- optimization ---
OPTIMIZER          = "adam"
LEARNING_RATE      = 5e-5
BATCH_SIZE         = 256
MAX_EPOCHS         = 100

# --- loss / output ---
LOSS               = "mse"     # demand is count data, should try mae and mse too
OUTPUT_UNITS       = 1
OUTPUT_ACTIVATION  = "softplus"        # non-negative expected count

# --- layer defaults ---
HIDDEN_ACTIVATION  = "relu"
WEIGHT_INIT        = "he_normal"

# --- regularization (off by default to isolate the complexity effect) ---
DROPOUT            = 0.0
WEIGHT_DECAY       = 0.0

# --- early stopping ---
EARLY_STOPPING     = True
MONITOR            = "val_loss"
PATIENCE           = 10

# --- data handling ---
SPLIT              = "random"
SCALER_FIT_ON      = "train_only"

# --- hex embedding ---
HEX_EMBED_DIM      = 16               # learned embedding dim per hexagon

# --- device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- reproducibility ---
SEEDS              = (0, 1, 2, 3, 4)   # run each model across all seeds; report mean +/- std

# --- architectures (the ONLY thing that varies across the 3 models) ---
# format: (n_hidden_layers, width)
ARCH_BASELINE      = (2, 64)
ARCH_WIDER         = (2, 128)          # depth fixed, width up
ARCH_DEEPER        = (4, 64)           # width fixed, depth up
# expose as module-like object so model code can use config.XXX
config = types.SimpleNamespace(
    LOSS           = LOSS,
    LEARNING_RATE  = LEARNING_RATE,
    WEIGHT_DECAY   = WEIGHT_DECAY,
    BATCH_SIZE     = BATCH_SIZE,
    MAX_EPOCHS     = MAX_EPOCHS,
    EARLY_STOPPING = EARLY_STOPPING,
    PATIENCE       = PATIENCE,
    OUTPUT_UNITS   = OUTPUT_UNITS,
    HEX_EMBED_DIM  = HEX_EMBED_DIM,
)

ARCH_NAMES = {ARCH_BASELINE: "baseline", ARCH_WIDER: "wide", ARCH_DEEPER: "deep"}

# per-arch best LRs from LR search; falls back to config.LEARNING_RATE if not set
BEST_LR = {}

# --- hp search (random search over regularization + architecture hyperparams) ---
LR_CANDIDATES      = [2e-5, 3e-5, 4e-5, 5e-5, 6e-5, 7e-5, 1e-4]
LR_SEARCH_SEEDS    = (0, 1, 2)
HP_DROPOUT_CANDIDATES  = [0.0]              # fixed: regularisation never helped
HP_WD_CANDIDATES       = [0.0]              # fixed: regularisation never helped
HP_EMBED_CANDIDATES    = [8, 16, 32]
HP_SEARCH_SEEDS        = (0, 1, 2, 3, 4)             # 1 seed sufficient for ranking combos
HP_PATIENCE            = 10               # lower patience during search; final runs use PATIENCE=10
BEST_HP                = {}              # {arch: {dropout, weight_decay, embed_dim}}


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Model + Training Utilities              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

class DemandBaseline(nn.Module):
    def __init__(self, input_dim, n_layers, width, n_hex, embed_dim, dropout=0.0):
        super().__init__()
        self.hex_embed = nn.Embedding(n_hex, embed_dim)
        nn.init.normal_(self.hex_embed.weight, std=0.01)

        layers = []
        in_dim = input_dim + embed_dim
        for _ in range(n_layers):
            linear = nn.Linear(in_dim, width)
            nn.init.kaiming_normal_(linear.weight, nonlinearity="relu")
            nn.init.zeros_(linear.bias)
            layers += [linear, nn.ReLU()]
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            in_dim = width
        out = nn.Linear(in_dim, config.OUTPUT_UNITS)
        nn.init.kaiming_normal_(out.weight, nonlinearity="relu")
        nn.init.zeros_(out.bias)
        layers += [out, nn.Softplus()]
        self.net = nn.Sequential(*layers)

    def forward(self, x, hex_idx):
        emb = self.hex_embed(hex_idx)          # (batch, embed_dim)
        x   = torch.cat([x, emb], dim=-1)      # (batch, input_dim + embed_dim)
        return self.net(x).squeeze(-1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_model(arch, X_train, y_train, X_val, y_val,
                hex_train, hex_val,
                seed=0, device="cpu", verbose=True, lr=None,
                dropout=0.0, weight_decay=None, batch_size=None, embed_dim=None,
                patience=None):
    set_seed(seed)
    n_layers, width = arch
    _embed_dim  = embed_dim    if embed_dim    is not None else config.HEX_EMBED_DIM
    _batch_size = batch_size   if batch_size   is not None else config.BATCH_SIZE
    _weight_dec = weight_decay if weight_decay is not None else config.WEIGHT_DECAY
    _patience   = patience   if patience   is not None else config.PATIENCE
    model = DemandBaseline(
        X_train.shape[1], n_layers, width, N_HEX, _embed_dim,
        dropout=dropout,
    ).to(device)

    _loss_name = getattr(config, "LOSS", "poisson")
    if _loss_name == "mse":
        loss_fn = nn.MSELoss()
    elif _loss_name == "mae":
        loss_fn = nn.L1Loss()
    else:
        loss_fn = nn.PoissonNLLLoss(log_input=False, full=False)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr if lr is not None else config.LEARNING_RATE,
        weight_decay=_weight_dec,
    )

    train_dl = DataLoader(
        TensorDataset(
            torch.as_tensor(X_train,   dtype=torch.float32),
            torch.as_tensor(hex_train, dtype=torch.long),
            torch.as_tensor(y_train,   dtype=torch.float32),
        ),
        batch_size=_batch_size,
        shuffle=True,
    )
    X_val_t   = torch.as_tensor(X_val,   dtype=torch.float32).to(device)
    hex_val_t = torch.as_tensor(hex_val, dtype=torch.long).to(device)
    y_val_t   = torch.as_tensor(y_val,   dtype=torch.float32).to(device)

    best_val, best_state, wait = float("inf"), None, 0
    epoch_width = len(str(config.MAX_EPOCHS))

    for epoch in range(config.MAX_EPOCHS):
        # ── train ──────────────────────────────────────────────────────────────
        model.train()
        running_loss, n_batches = 0.0, 0
        for xb, hb, yb in train_dl:
            xb, hb, yb = xb.to(device), hb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb, hb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches    += 1
        train_loss = running_loss / n_batches

        # ── validate ───────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t, hex_val_t), y_val_t).item()

        # ── early stopping ─────────────────────────────────────────────────────
        improved = val_loss < best_val
        if improved:
            best_val, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1

        if verbose:
            marker = " *" if improved else f" (no improvement {wait}/{_patience})"
            print(f"  epoch {epoch+1:{epoch_width}d}/{config.MAX_EPOCHS}"
                  f"  train={train_loss:.4f}  val={val_loss:.4f}{marker}")

        if config.EARLY_STOPPING and wait >= _patience:
            if verbose:
                print(f"  early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    return model, best_val


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# HP Search Utility                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def run_hp_search(arch, X_train, y_train, X_val, y_val,
                  hex_train, hex_val,
                  device='cpu',
                  dropout_candidates=None, wd_candidates=None,
                  embed_candidates=None, seeds=None):
    import itertools

    dropout_candidates = dropout_candidates or HP_DROPOUT_CANDIDATES
    wd_candidates      = wd_candidates      or HP_WD_CANDIDATES
    embed_candidates   = embed_candidates   or HP_EMBED_CANDIDATES
    seeds              = seeds              or HP_SEARCH_SEEDS
    arch_label         = ARCH_NAMES.get(arch, str(arch))
    best_lr            = BEST_LR.get(arch, config.LEARNING_RATE)

    combos = list(itertools.product(
        dropout_candidates, wd_candidates, embed_candidates
    ))

    print(f'HP Search — {arch_label} (lr={best_lr:.0e})')
    print(f'{len(combos)} combos × {len(seeds)} seed(s)'
          f' = {len(combos) * len(seeds)} runs  |  patience={HP_PATIENCE}\n')
    print(f"{'#':>4}  {'dropout':>8}  {'wd':>8}  {'emb':>5}  "
          f"{'mean val':>10}  {'std':>8}")
    print('-' * 50)

    results = []
    for i, (dropout, wd, emb) in enumerate(combos, 1):
        val_losses = []
        for seed in seeds:
            _, val_loss = train_model(
                arch, X_train, y_train, X_val, y_val,
                hex_train, hex_val,
                seed=seed, device=device, verbose=False,
                lr=best_lr, dropout=dropout, weight_decay=wd,
                embed_dim=emb, patience=HP_PATIENCE,
            )
            val_losses.append(val_loss)
        mean_loss = float(np.mean(val_losses))
        std_loss  = float(np.std(val_losses))
        results.append({
            'dropout': dropout, 'weight_decay': wd,
            'embed_dim': emb,
            'mean_val_loss': mean_loss, 'std_val_loss': std_loss,
        })
        print(f'{i:>4}  {dropout:>8.2f}  {wd:>8.1e}  {emb:>5}  '
              f'{mean_loss:>10.4f}  {std_loss:>8.4f}')

    results_df = pd.DataFrame(results).sort_values('mean_val_loss').reset_index(drop=True)
    print(f'\n--- Top 5 ---')
    print(results_df.head(5).to_string(index=False))

    best = results_df.iloc[0]
    BEST_HP[arch] = {
        'dropout':      float(best['dropout']),
        'weight_decay': float(best['weight_decay']),
        'embed_dim':    int(best['embed_dim']),
    }
    print(f'\nBEST_HP[{arch_label}] = {BEST_HP[arch]}')
    return BEST_HP[arch]


## Model Architecture — Data Flow

```
One row of raw data
┌─────────────────┬──────────────────────────────┬─────────────┐
│ pickup_h3_res7  │  is_weekend, temp, rain, ...  │ trip_count  │
│ '872664190fff'  │  0,  12.3,  0.0,  ...        │      5      │
└────────┬────────┴───────────────┬───────────────┴──────┬──────┘
         │                       │                       │
         ▼                       ▼                       ▼
    hex_vocab              StandardScaler             target y
  '872664190fff'          [0.0, -0.3, ...]              5.0
       → 42
         │                       │
         ▼                       │
  Embedding table                │
  (600 hexagons × 16 dims)       │
  row 42: [0.12, -0.31, ...]     │
     (16 dims, learned)          │ (29 dims, scaled)
         │                       │
         └───────────┬───────────┘
                     ▼
               torch.cat(...)
          [0.12, -0.31, ..., 0.0, -0.3, ...]
                  (16 + 29 = 45 dims)
                     │
                     ▼
           ┌─────────────────────┐
           │  Linear(45 → 64)    │
           │  ReLU               │
           │  Linear(64 → 64)    │  ← baseline / deeper adds more blocks
           │  ReLU               │
           │  Linear(64 → 1)     │
           │  Softplus           │  ← keeps output ≥ 0 (trip count)
           └─────────────────────┘
                     │
                     ▼
              ŷ  (predicted trip count)
```

The embedding table starts with near-zero weights and is updated by backprop
alongside all other parameters — the network learns which hexagons are similar.


## Train / Val / Test Split

We split rows randomly into **70 % train / 15 % val / 15 % test** using
`train_test_split` with a fixed random state for reproducibility.

| Split | Share |
|-------|-------|
| Train | ~70 % |
| Val   | ~15 % |
| Test  | ~15 % |


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Train / Val / Test Split                #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

data['time_bucket'] = pd.to_datetime(data['time_bucket'], format='mixed')

# 70 / 15 / 15 random split
train_data, temp_data = train_test_split(data, test_size=0.30, random_state=42, shuffle=False)
val_data,   test_data = train_test_split(temp_data, test_size=0.50, random_state=42, shuffle=False)

train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)
test_data  = test_data.reset_index(drop=True)

print(f"Train : {len(train_data):>9,} rows")
print(f"Val   : {len(val_data):>9,} rows")
print(f"Test  : {len(test_data):>9,} rows")


## Feature Preparation

Drop leakage columns (trip-derived aggregates from the same time-bucket),
ID/index columns, categorical columns not yet encoded, and the target.  
Fit a `StandardScaler` on the **training set only** to avoid leakage into val/test.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Feature Preparation                     #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# trip-derived stats (same time-bucket → leakage) + categoricals not yet encoded
LEAKAGE_COLS = [
    'active_taxis', 'avg_idle_time', 'avg_trip_duration', 'avg_trip_distance',
    'avg_fare', 'avg_trip_total', 'avg_tip', 'tip_rate', 'share_cash_payment',
    'area_type', 'season',
]

# raw cyclic integers replaced by sin/cos encodings
RAW_CYCLIC_COLS = ['month', 'hour_of_day', 'day_of_week']

ID_COLS    = ['time_bucket', 'bucket_index', 'pickup_h3_res6']
TARGET_COL = 'trip_count'

FEATURE_COLS = [
    c for c in train_data.columns
    if c not in LEAKAGE_COLS + RAW_CYCLIC_COLS + ID_COLS + [TARGET_COL]
]
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_data[FEATURE_COLS].values)
X_val   = scaler.transform(val_data[FEATURE_COLS].values)
X_test  = scaler.transform(test_data[FEATURE_COLS].values)

y_train = np.log1p(train_data[TARGET_COL].values.astype(float))
y_val   = np.log1p(val_data[TARGET_COL].values.astype(float))
y_test  = test_data[TARGET_COL].values.astype(float)  # kept raw for metric reporting

# hex embedding indices — vocabulary built from training set
hex_vocab = {h: i for i, h in enumerate(sorted(train_data['pickup_h3_res6'].unique()))}
N_HEX     = len(hex_vocab)

hex_train = train_data['pickup_h3_res6'].map(hex_vocab).values
hex_val   = val_data['pickup_h3_res6'].map(hex_vocab).values
hex_test  = test_data['pickup_h3_res6'].map(hex_vocab).values

print(f"\nX_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")
print(f"\nUnique hexagons : {N_HEX}  |  embed dim : {HEX_EMBED_DIM}")


## Benchmark Models

Before diving into the three NN architectures, we evaluate two simple non-neural baselines on the held-out test set:

1. **Historical-mean predictor** — for each hexagon × hour-of-day pair, predict the mean `trip_count` observed in the training split. Unseen combinations fall back to the global training mean.
2. **Ridge regression** — a linear model on the same scaled feature matrix (`X_train`). Trained on log₁⁺ demand and back-transformed with `expm1` at evaluation time, matching the NN target encoding.

These numbers set the floor: any NN that cannot beat Ridge is not adding value.

Metrics match the NN evaluation cells:
- **R²** — coefficient of determination
- **MAE** — mean absolute error (in raw trip counts)
- **RMSE** — root mean squared error
- **NRMSE** — RMSE normalised by mean actual demand (scale-free)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Benchmark Models                        #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = 100.0 * zero_demand_count / len(y_test)
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

print(f"Zero-demand rows : {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")


def _eval_benchmark(label, preds):
    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot
    print(f"{label:<35s}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  NRMSE={nrmse:.4f}")
    return dict(r2=round(r2, 6), mae=round(mae, 6), rmse=round(rmse, 6), nrmse=round(nrmse, 6))


# ── 1. Historical-mean predictor ───────────────────────────────────────────────
mean_table = (
    train_data.groupby(['pickup_h3_res6', 'hour_of_day'])['trip_count']
    .mean()
    .rename('pred_mean')
)
global_mean = float(train_data['trip_count'].mean())

test_lookup      = test_data[['pickup_h3_res6', 'hour_of_day']].copy()
test_lookup      = test_lookup.join(mean_table, on=['pickup_h3_res6', 'hour_of_day'])
hist_mean_preds  = test_lookup['pred_mean'].fillna(global_mean).values

bm_hist = _eval_benchmark("Historical mean (hex × hour)", hist_mean_preds)

# ── 2. Ridge regression ────────────────────────────────────────────────────────
ridge        = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)            # y_train is log1p-transformed
ridge_preds  = np.expm1(ridge.predict(X_test))
ridge_preds  = np.maximum(ridge_preds, 0.0)   # clamp to non-negative

bm_ridge = _eval_benchmark("Ridge regression", ridge_preds)


## Baseline Model — Training

Run `ARCH_BASELINE = (2 hidden layers, 64 units)` across all seeds and report
the mean ± std validation loss.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Visualization          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

input_dim = X_train.shape[1]
n_layers, width = ARCH_BASELINE

viz_model = DemandBaseline(input_dim, n_layers, width, N_HEX, HEX_EMBED_DIM)
x_dummy   = torch.zeros(1, input_dim)
h_dummy   = torch.zeros(1, dtype=torch.long)
summary(viz_model, input_data=(x_dummy, h_dummy), col_names=["input_size", "output_size", "num_params"], verbose=1)


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Training               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

print(f"Device: {device}\n")

baseline_val_losses  = []
baseline_models      = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_BASELINE, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
    )
    baseline_val_losses.append(val_loss)
    baseline_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nBaseline  val loss:  {np.mean(baseline_val_losses):.4f} ± {np.std(baseline_val_losses):.4f}")


## Baseline Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **Zero-demand rows** — share of test rows with `trip_count = 0` (context for inflated R²)
- **Mean Actual Test Demand** — average true trip count in the test set
- **R²** — coefficient of determination (share of variance explained)
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **NRMSE** — RMSE normalised by mean actual demand (scale-free)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Baseline Model — Evaluation             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results  = []
r2_scores, mae_scores, rmse_scores, nrmse_scores = [], [], [], []
n_layers, width = ARCH_BASELINE

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct = 100.0 * zero_demand_count / len(y_test)
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

print(f"Zero-demand rows: {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")

for seed, model in zip(SEEDS, baseline_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()
        preds = np.expm1(preds)

    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot

    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    nrmse_scores.append(nrmse)

    print(f"  seed={seed}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  NRMSE={nrmse:.4f}")

    run_results.append({
        "timestamp"         : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"             : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"          : n_layers,
        "width"             : width,
        "learning_rate"     : BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
        "batch_size"        : BATCH_SIZE,
        "seed"              : seed,
        "val_loss"          : baseline_val_losses[seed],
        "zero_demand_count" : zero_demand_count,
        "mean_actual"       : round(mean_actual, 6),
        "r2"                : round(r2,    6),
        "mae"               : round(mae,   6),
        "rmse"              : round(rmse,  6),
        "nrmse"             : round(nrmse, 6),
    })

arch_label = ARCH_NAMES.get((n_layers, width), "model")
print(f"\n{arch_label.capitalize()}  R²    : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
print(f"{arch_label.capitalize()}  MAE   : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"{arch_label.capitalize()}  RMSE  : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"{arch_label.capitalize()}  NRMSE : {np.mean(nrmse_scores):.4f} ± {np.std(nrmse_scores):.4f}")

Tuned Baseline  R²=0.9313 ± 0.0020  MAE=4.4563 ± 0.0101  NRMSE=0.7787 ± 0.0113


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


## Representation Probing — Baseline Model

**Question**: can the model infer the specific calendar **date** from its input
features? If yes, random train/test splitting is unsafe — same-date rows
(identical weather, same day-of-week) end up in both train and test, and the
model can exploit that fingerprint rather than learning to generalise.

**Protocol**

1. Register a forward hook on the **last ReLU** (64-dim layer) and extract
   activations for all train and test rows.
2. Derive sequential date targets from `time_bucket` (all strip the hour so
   probing is purely about which calendar period it is):
   - `date_index` — days since first date in dataset
   - `week_index` — weeks since first date in dataset
3. For each target fit a `Ridge` linear probe and compare three probes:

   | Probe | What it measures |
   |---|---|
   | **Cyclic-only probe** | Floor: sin/cos time features — cannot distinguish e.g. week 1 from week 52 |
   | **Full raw-X probe** | Ceiling: all 29 scaled inputs including weather |
   | **Repr probe** | What the 64-dim hidden activations encode |

R²(repr) − R²(cyclic floor) > 0.10 → model encodes specific dates → chronological split needed.

**Sanity check**: re-running with shuffled labels must collapse R² to ~0.

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Representation-Probing Test             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score as _r2

# Cyclic feature column names (sin/cos encodings already in X)
CYCLIC_COLS = [c for c in FEATURE_COLS
               if any(c.startswith(p) for p in
                      ('hour_of_day_', 'day_of_week_', 'month_'))]
CYCLIC_IDX  = [FEATURE_COLS.index(c) for c in CYCLIC_COLS]

# Probe targets: days / weeks since first date in dataset (no time component)
_min_date  = data['time_bucket'].dt.normalize().min()
_days_tr   = (train_data['time_bucket'].dt.normalize() - _min_date).dt.days
_days_te   = (test_data['time_bucket'].dt.normalize()  - _min_date).dt.days

PROBE_TARGETS = {
    'date_index' : (_days_tr.values.astype(float),       _days_te.values.astype(float)),
    'week_index' : ((_days_tr // 7).values.astype(float), (_days_te // 7).values.astype(float)),
}


def extract_hidden_reps(model, X, hex_idx, device, batch_size=4096):
    model.eval()
    captured = []

    last_relu = None
    for layer in model.net:
        if isinstance(layer, nn.ReLU):
            last_relu = layer

    def _hook(_, __, output):
        captured.append(output.detach().cpu())

    handle = last_relu.register_forward_hook(_hook)
    X_t = torch.as_tensor(X,       dtype=torch.float32)
    h_t = torch.as_tensor(hex_idx, dtype=torch.long)

    with torch.no_grad():
        for start in range(0, X.shape[0], batch_size):
            model(
                X_t[start : start + batch_size].to(device),
                h_t[start : start + batch_size].to(device),
            )

    handle.remove()
    return np.vstack([t.numpy() for t in captured])


# Use seed-0 baseline model (trained in cell above)
probe_model = baseline_models[0]

print('Extracting hidden representations ...')
reps_train = extract_hidden_reps(probe_model, X_train, hex_train, device)
reps_test  = extract_hidden_reps(probe_model, X_test,  hex_test,  device)
print(f'  train reps: {reps_train.shape}   test reps: {reps_test.shape}\n')

rng = np.random.default_rng(42)

for target_name, (y_tr, y_te) in PROBE_TARGETS.items():
    r2_cyclic = _r2(y_te, Ridge(alpha=1.0).fit(X_train[:, CYCLIC_IDX], y_tr)
                                            .predict(X_test[:, CYCLIC_IDX]))
    r2_raw    = _r2(y_te, Ridge(alpha=1.0).fit(X_train, y_tr).predict(X_test))
    r2_repr   = _r2(y_te, Ridge(alpha=1.0).fit(reps_train, y_tr).predict(reps_test))
    r2_shuf   = _r2(y_te, Ridge(alpha=1.0).fit(reps_train, rng.permutation(y_tr))
                                            .predict(reps_test))

    above_floor = r2_repr - r2_cyclic
    verdict = ('-> chronological split recommended'
               if above_floor > 0.10 else '-> random split acceptable')

    print(f'Target: {target_name}')
    print(f'  Cyclic-only probe  R2 = {r2_cyclic:.4f}  <- floor')
    print(f'  Full raw-X probe   R2 = {r2_raw:.4f}  <- ceiling')
    print(f'  Repr probe         R2 = {r2_repr:.4f}')
    print(f'  Shuffled (sanity)  R2 = {r2_shuf:.4f}  <- should be ~0')
    print(f'  Delta (repr - floor) = {above_floor:+.4f}  {verdict}')
    print()


## Hyperparameter Search — Baseline Model

Grid search over 4 hyperparameters using the best LR from the LR search above.

| Hyperparameter | Candidates |
|---|---|
| `dropout`      | 0.0, 0.1, 0.2, 0.3 |
| `weight_decay` | 0.0, 1e-5, 1e-4, 1e-3 |
| `embed_dim`    | 8, 16, 32 |

Full grid: 4 × 4 × 3 = **48 combinations** × 1 seed = **48 training runs**.  
The best combination is stored in `BEST_HP[ARCH_BASELINE]`.


In [ ]:
run_hp_search(
    ARCH_BASELINE,
    X_train, y_train, X_val, y_val,
    hex_train, hex_val,
    device=device,
)


## Tuned Baseline — Training

Re-train the baseline architecture with the best hyperparameters found above,
across all `SEEDS` for a stable mean ± std estimate.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Tuned Baseline — Training               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

hp = BEST_HP[ARCH_BASELINE]
print(f"Best HP: {hp}\n")

tuned_val_losses = []
tuned_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_BASELINE, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
        dropout=hp["dropout"],
        weight_decay=hp["weight_decay"],
        embed_dim=hp["embed_dim"],
    )
    tuned_val_losses.append(val_loss)
    tuned_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nTuned Baseline  val loss:  "
      f"{np.mean(tuned_val_losses):.4f} ± {np.std(tuned_val_losses):.4f}")


## Tuned Baseline — Evaluation

Evaluate on the held-out **test set** and compare against the untuned baseline.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Tuned Baseline — Evaluation             #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results  = []
r2_scores, mae_scores, rmse_scores, nrmse_scores = [], [], [], []
n_layers, width = ARCH_BASELINE

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = 100.0 * zero_demand_count / len(y_test)
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

print(f"Zero-demand rows: {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")

for seed, model in zip(SEEDS, tuned_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()
        preds = np.expm1(preds)

    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot

    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    nrmse_scores.append(nrmse)

    print(f"  seed={seed}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  NRMSE={nrmse:.4f}")

    run_results.append({
        "timestamp"         : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"             : "baseline_tuned",
        "n_layers"          : n_layers,
        "width"             : width,
        "learning_rate"     : BEST_LR.get(ARCH_BASELINE, config.LEARNING_RATE),
        "batch_size"        : BATCH_SIZE,
        "seed"              : seed,
        "val_loss"          : tuned_val_losses[seed],
        "zero_demand_count" : zero_demand_count,
        "mean_actual"       : mean_actual,
        "r2"                : r2,
        "mae"               : mae,
        "rmse"              : rmse,
        "nrmse"             : nrmse,
    })

print(f"\nTuned Baseline  "
      f"R²={np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}  "
      f"MAE={np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}  "
      f"NRMSE={np.mean(nrmse_scores):.4f} ± {np.std(nrmse_scores):.4f}")


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


## Deeper Model — Training

Run `ARCH_DEEPER = (4 hidden layers, 64 units)` across all seeds.
Width is fixed; depth doubles relative to the baseline.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Deeper Model — Training                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

deeper_val_losses = []
deeper_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_DEEPER, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_DEEPER, config.LEARNING_RATE),
    )
    deeper_val_losses.append(val_loss)
    deeper_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nDeeper  val loss:  {np.mean(deeper_val_losses):.4f} ± {np.std(deeper_val_losses):.4f}")


## Deeper Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **Zero-demand rows** — share of test rows with `trip_count = 0` (context for inflated R²)
- **Mean Actual Test Demand** — average true trip count in the test set
- **R²** — coefficient of determination (share of variance explained)
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **NRMSE** — RMSE normalised by mean actual demand (scale-free)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Deeper   Model — Evaluation               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results  = []
r2_scores, mae_scores, rmse_scores, nrmse_scores = [], [], [], []
n_layers, width = ARCH_DEEPER

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = zero_demand_count / len(y_test) * 100
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

print(f"Zero-demand rows: {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")

for seed, model in zip(SEEDS, deeper_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()
        preds = np.expm1(preds)

    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot

    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    nrmse_scores.append(nrmse)

    print(f"  seed={seed}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  NRMSE={nrmse:.4f}")

    run_results.append({
        "timestamp"         : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"             : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"          : n_layers,
        "width"             : width,
        "learning_rate"     : BEST_LR.get(ARCH_DEEPER, config.LEARNING_RATE),
        "batch_size"        : BATCH_SIZE,
        "seed"              : seed,
        "val_loss"          : deeper_val_losses[seed],
        "zero_demand_count" : zero_demand_count,
        "zero_demand_pct"   : round(zero_demand_pct, 2),
        "mean_actual"       : round(mean_actual, 6),
        "r2"                : round(r2,    6),
        "mae"               : round(mae,   6),
        "rmse"              : round(rmse,  6),
        "nrmse"             : round(nrmse, 6),
    })

arch_label = ARCH_NAMES.get((n_layers, width), "model")
print(f"\n{arch_label.capitalize()}  R²    : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
print(f"{arch_label.capitalize()}  MAE   : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"{arch_label.capitalize()}  RMSE  : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"{arch_label.capitalize()}  NRMSE : {np.mean(nrmse_scores):.4f} ± {np.std(nrmse_scores):.4f}")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


## Hyperparameter Search — Deeper Model

Same grid search as for the baseline, run on `ARCH_DEEPER = (4 hidden layers, 64 units)`.
Result is stored in `BEST_HP[ARCH_DEEPER]`.

| Hyperparameter | Candidates |
|---|---|
| `dropout`      | 0.0, 0.1, 0.2, 0.3 |
| `weight_decay` | 0.0, 1e-5, 1e-4, 1e-3 |
| `embed_dim`    | 8, 16, 32 |

Full grid: 4 × 4 × 3 = **48 combinations** × 1 seed = **48 training runs**.  
The best combination is stored in `BEST_HP[ARCH_DEEPER]`.

In [ ]:
run_hp_search(
    ARCH_DEEPER,
    X_train, y_train, X_val, y_val,
    hex_train, hex_val,
    device=device,
)


## LR Search — Wider Model

Same grid search as for the baseline, run on `ARCH_WIDER = (2 hidden layers, 128 units)`.
Result is stored in `BEST_LR[ARCH_WIDER]`.


In [ ]:
run_lr_search(ARCH_WIDER,    X_train, y_train, X_val, y_val, hex_train, hex_val, device=device)  # (2, 128)


## Wider Model — Training

Run `ARCH_WIDER = (2 hidden layers, 128 units)` across all seeds.
Depth is fixed; width doubles relative to the baseline.


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Wider Model — Training                  #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

wider_val_losses = []
wider_models     = []

for seed in SEEDS:
    model, val_loss = train_model(
        ARCH_WIDER, X_train, y_train, X_val, y_val,
        hex_train, hex_val,
        seed=seed, device=device,
        lr=BEST_LR.get(ARCH_WIDER, config.LEARNING_RATE),
    )
    wider_val_losses.append(val_loss)
    wider_models.append(model)
    print(f"  seed={seed}  val_loss={val_loss:.4f}")

print(f"\nWider  val loss:  {np.mean(wider_val_losses):.4f} ± {np.std(wider_val_losses):.4f}")


## Wider Model — Evaluation

Evaluate on the held-out **test set** using the best-checkpoint model from each
seed, then report mean ± std across seeds.

Metrics:
- **Zero-demand rows** — share of test rows with `trip_count = 0` (context for inflated R²)
- **Mean Actual Test Demand** — average true trip count in the test set
- **R²** — coefficient of determination (share of variance explained)
- **MAE** — mean absolute error (interpretable in trip counts)
- **RMSE** — root mean squared error (penalises large misses more)
- **NRMSE** — RMSE normalised by mean actual demand (scale-free)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Wider    Model — Evaluation               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import datetime

X_test_t   = torch.as_tensor(X_test,   dtype=torch.float32).to(device)
hex_test_t = torch.as_tensor(hex_test, dtype=torch.long).to(device)

run_results  = []
r2_scores, mae_scores, rmse_scores, nrmse_scores = [], [], [], []
n_layers, width = ARCH_WIDER

zero_demand_count = int((y_test == 0).sum())
zero_demand_pct   = zero_demand_count / len(y_test) * 100
mean_actual       = float(y_test.mean())
ss_tot            = float(np.sum((y_test - mean_actual) ** 2))

print(f"Zero-demand rows: {zero_demand_count:,} / {len(y_test):,} ({zero_demand_pct:.1f}%)")
print(f"Mean actual test demand : {mean_actual:.4f}\n")

for seed, model in zip(SEEDS, wider_models):
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t, hex_test_t).cpu().numpy()
        preds = np.expm1(preds)

    mae   = float(np.mean(np.abs(preds - y_test)))
    rmse  = float(np.sqrt(np.mean((preds - y_test) ** 2)))
    nrmse = rmse / mean_actual if mean_actual > 0 else float("nan")
    r2    = 1.0 - float(np.sum((preds - y_test) ** 2)) / ss_tot

    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    nrmse_scores.append(nrmse)

    print(f"  seed={seed}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  NRMSE={nrmse:.4f}")

    run_results.append({
        "timestamp"         : datetime.datetime.now().isoformat(timespec="seconds"),
        "model"             : ARCH_NAMES.get((n_layers, width), "unknown"),
        "n_layers"          : n_layers,
        "width"             : width,
        "learning_rate"     : BEST_LR.get(ARCH_WIDER, config.LEARNING_RATE),
        "batch_size"        : BATCH_SIZE,
        "seed"              : seed,
        "val_loss"          : wider_val_losses[seed],
        "zero_demand_count" : zero_demand_count,
        "zero_demand_pct"   : round(zero_demand_pct, 2),
        "mean_actual"       : round(mean_actual, 6),
        "r2"                : round(r2,    6),
        "mae"               : round(mae,   6),
        "rmse"              : round(rmse,  6),
        "nrmse"             : round(nrmse, 6),
    })

arch_label = ARCH_NAMES.get((n_layers, width), "model")
print(f"\n{arch_label.capitalize()}  R²    : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")
print(f"{arch_label.capitalize()}  MAE   : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"{arch_label.capitalize()}  RMSE  : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"{arch_label.capitalize()}  NRMSE : {np.mean(nrmse_scores):.4f} ± {np.std(nrmse_scores):.4f}")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save Results                            #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

RESULTS_PATH = "data/results/nn_results.csv"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

results_df = pd.DataFrame(run_results)

if os.path.exists(RESULTS_PATH):
    results_df.to_csv(RESULTS_PATH, mode="a", header=False, index=False)
    print(f"Appended {len(results_df)} rows to {RESULTS_PATH}")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"Created {RESULTS_PATH} with {len(results_df)} rows")

print(pd.read_csv(RESULTS_PATH).tail(len(results_df)).to_string(index=False))


## Hyperparameter Search — Wider Model

Same grid search as for the baseline, run on `ARCH_WIDER = (2 hidden layers, 128 units)`.
Result is stored in `BEST_HP[ARCH_WIDER]`.

| Hyperparameter | Candidates |
|---|---|
| `dropout`      | 0.0, 0.1, 0.2, 0.3 |
| `weight_decay` | 0.0, 1e-5, 1e-4, 1e-3 |
| `embed_dim`    | 8, 16, 32 |

Full grid: 4 × 4 × 3 = **48 combinations** × 1 seed = **48 training runs**.  
The best combination is stored in `BEST_HP[ARCH_WIDER]`.

In [ ]:
run_hp_search(
    ARCH_WIDER,
    X_train, y_train, X_val, y_val,
    hex_train, hex_val,
    device=device,
)
